# CHAMELEON layout distillation (LoRA on Qwen3-4B-Instruct)

Teaches a small open model to lay out a resized ad the way Claude Sonnet does. It reads the source ad as JSON plus a target canvas and writes every element's new box as JSON. The engine then fits the text, checks the hard rules and falls back to the algorithm when a plan fails.

**Before you run**
1. **Runtime → Change runtime type → L4 GPU** (A100 also works; T4 works but is about 3x slower).
2. In Google Drive, make a folder **`chameleon-distill`** and upload **`train.jsonl`** and **`eval_prompts.jsonl`** into it (from `distil-research/training-data/llm-distill/`).
3. **Runtime → Run all.** It asks once for permission to mount Drive.

**If the runtime disconnects:** reconnect and press **Run all** again. Training resumes from the last checkpoint in `chameleon-distill/checkpoints`; finished steps are not redone.

**What you get back (all in `chameleon-distill/`):** `eval_outputs.jsonl` (send this one back), `lora_adapter/`, `gguf/` (the offline model file), `train_log.json`.

In [ ]:
# 1. Drive + paths
from google.colab import drive
drive.mount('/content/drive')
import os, json, glob
D = '/content/drive/MyDrive/chameleon-distill'
assert os.path.exists(f'{D}/train.jsonl'), 'upload train.jsonl to MyDrive/chameleon-distill first'
CKPT = f'{D}/checkpoints'; os.makedirs(CKPT, exist_ok=True)
BASE = 'unsloth/Qwen3-4B-Instruct-2507'   # change this one line to retrain on another approved base
MAX_SEQ = 8192
print(sum(1 for _ in open(f'{D}/train.jsonl')), 'training examples')

In [ ]:
# 2. Install (a few minutes)
!pip -q install unsloth
import unsloth, trl, transformers, peft
print('unsloth', unsloth.__version__, '| trl', trl.__version__, '| transformers', transformers.__version__, '| peft', peft.__version__)

In [ ]:
# 3. Load the base model in 4-bit and add a LoRA adapter
from unsloth import FastLanguageModel
model, tokenizer = FastLanguageModel.from_pretrained(BASE, max_seq_length=MAX_SEQ, load_in_4bit=True)
model = FastLanguageModel.get_peft_model(
    model, r=16, lora_alpha=16, lora_dropout=0, bias='none',
    target_modules=['q_proj','k_proj','v_proj','o_proj','gate_proj','up_proj','down_proj'],
    use_gradient_checkpointing='unsloth', random_state=0)

In [ ]:
# 4. Dataset: system + user prompt -> assistant layout JSON (loss on the answer only)
from datasets import Dataset
SYSTEM = 'You are an ad layout engine. Given a source ad as JSON and a target canvas, return the resized layout as JSON.'
rows = [json.loads(l) for l in open(f'{D}/train.jsonl')]
def to_text(r):
    msgs = [{'role':'system','content':SYSTEM},{'role':'user','content':r['prompt']},{'role':'assistant','content':r['answer']}]
    return tokenizer.apply_chat_template(msgs, tokenize=False)
texts = [to_text(r) for r in rows]
lens = [len(tokenizer(t).input_ids) for t in texts]
keep = [t for t, n in zip(texts, lens) if n <= MAX_SEQ]
print(f'{len(keep)}/{len(texts)} examples fit {MAX_SEQ} tokens; longest {max(lens)}')
ds = Dataset.from_dict({'text': keep}).shuffle(seed=0)
split = ds.train_test_split(test_size=0.05, seed=0)

In [ ]:
# 5. Train (checkpoints go to Drive; a re-run resumes)
import inspect
from trl import SFTTrainer, SFTConfig
from unsloth.chat_templates import train_on_responses_only
cfg = dict(
    output_dir=CKPT, dataset_text_field='text',
    per_device_train_batch_size=1, gradient_accumulation_steps=8,
    num_train_epochs=3, learning_rate=2e-4, lr_scheduler_type='cosine', warmup_ratio=0.05,
    optim='adamw_8bit', bf16=True, logging_steps=5,
    save_strategy='steps', save_steps=25, save_total_limit=3,
    eval_strategy='steps', eval_steps=25, report_to='none', seed=0)
# trl renamed max_seq_length -> max_length, and tokenizer -> processing_class; support both
cfg['max_length' if 'max_length' in inspect.signature(SFTConfig).parameters else 'max_seq_length'] = MAX_SEQ
args = SFTConfig(**cfg)
kw = dict(model=model, train_dataset=split['train'], eval_dataset=split['test'], args=args)
kw['processing_class' if 'processing_class' in inspect.signature(SFTTrainer.__init__).parameters else 'tokenizer'] = tokenizer
trainer = SFTTrainer(**kw)
trainer = train_on_responses_only(trainer, instruction_part='<|im_start|>user\n', response_part='<|im_start|>assistant\n')
resume = bool(glob.glob(f'{CKPT}/checkpoint-*'))
print('resuming from checkpoint' if resume else 'starting fresh')
trainer.train(resume_from_checkpoint=resume)
json.dump(trainer.state.log_history, open(f'{D}/train_log.json', 'w'))
model.save_pretrained(f'{D}/lora_adapter'); tokenizer.save_pretrained(f'{D}/lora_adapter')
print('saved adapter')

In [ ]:
# 6. Generate layouts for the 30 held-out prompts (send eval_outputs.jsonl back)
FastLanguageModel.for_inference(model)
out_path = f'{D}/eval_outputs.jsonl'
done = {json.loads(l)['cell'] for l in open(out_path)} if os.path.exists(out_path) else set()
with open(out_path, 'a') as f:
    for r in map(json.loads, open(f'{D}/eval_prompts.jsonl')):
        if r['cell'] in done: continue
        msgs = [{'role':'system','content':SYSTEM},{'role':'user','content':r['prompt']}]
        ids = tokenizer.apply_chat_template(msgs, add_generation_prompt=True, return_tensors='pt').to('cuda')
        gen = model.generate(input_ids=ids, max_new_tokens=3000, do_sample=False)
        text = tokenizer.decode(gen[0][ids.shape[1]:], skip_special_tokens=True)
        f.write(json.dumps({'cell': r['cell'], 'output': text}) + '\n'); f.flush()
        print(r['cell'], len(text), 'chars')
print('done ->', out_path)

In [ ]:
# 7. Export the offline model file (GGUF, 4-bit) for llama.cpp inside Docker
model.save_pretrained_gguf(f'{D}/gguf', tokenizer, quantization_method='q4_k_m')
print(os.listdir(f'{D}/gguf'))

## Optional, later: publish the weights
The rules need fine-tuned weights to be public (for example a public Hugging Face repo). Do this only after the held-out test shows it helps. Run `from huggingface_hub import notebook_login; notebook_login()`, paste **your own** token in the box, then `model.push_to_hub('<you>/chameleon-layout-qwen3-4b-lora')`.